In [1]:
import os
os.environ["RAY_DISABLED"] = "1"

# 安装 YOLOv8
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.4 MB/s eta 0:00:00


In [2]:
# 1. 打印 /kaggle/input/datasets/viviorangeani/ 下的所有文件夹
print("=== /kaggle/input/datasets/viviorangeani/ 目录内容 ===")
print(os.listdir("/kaggle/input/datasets/viviorangeani/"))

# 2. 打印 field_handheld 文件夹下的内容
print("\n=== /kaggle/input/datasets/viviorangeani/corn-hand415/ 目录内容 ===")
print(os.listdir("/kaggle/input/datasets/viviorangeani/corn-hand415"))

=== /kaggle/input/datasets/viviorangeani/ 目录内容 ===
['corn-hand415']

=== /kaggle/input/datasets/viviorangeani/corn-hand415/ 目录内容 ===
['labels', 'images', 'data_kaggle.yaml']


In [3]:
# 导入模型
from ultralytics import YOLO

# 加载 yolov8n 预训练模型
model = YOLO("yolov8n-obb.pt")

# 开始训练（30轮 + 完整数据增强 + 基础学习率）
model.train(
    # 你的数据集路径
    data="/kaggle/input/datasets/viviorangeani/corn-hand415/data_kaggle.yaml",
    task="obb",
    epochs=30,
    imgsz=800,
    batch=16,
    device=0,
    
    # ===================== 数据增强配置（满足你的全部要求）=====================
    # 1. 基础几何变换
    fliplr=0.5,          # 左右翻转（开启）
    flipud=0.2,          # 上下翻转（开启，概率适中）
    degrees=5.0,         # 旋转（±5°，小角度防止叶片跑出框）
    translate=0.05,      # 平移（5%，小比例，贴合你的要求）
    # shear=3.0,           # 剪切（±3°，温和增强）
    
    # 2. 颜色 / 光照增强 + 透视
    hsv_h=0.015,         # 色调增强（保留原配置）
    hsv_s=0.7,           # 饱和度增强（保留原配置）
    hsv_v=0.4,           # 亮度增强（保留原配置）
    # perspective=0.001,   # 透视变换（保留原配置）
    
    # 3. Mosaic 马赛克增强（开启）
    mosaic=1.0,          # 100% 启用 mosaic

    # ===================== 基础学习率（保持不变）=====================
    lr0=0.01,
    lrf=0.01,

    # ===================== 保存路径 =====================
    project="/kaggle/working/yolov8_train",
    name="corn_model",
    save=True,
    pretrained=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/viviorangeani/corn-hand415/data_kaggle.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hs

ultralytics.utils.metrics.OBBMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78815ef19d30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [4]:
# ===================== 自动输出 Baseline 指标 =====================
import numpy as np
results = model.val()  # 重新验证一遍（可选，也可以直接用训练结果）

# 提取全局平均指标（和mAP对应，符合报告规范）
map50 = results.box.map50
map50_95 = results.box.map
precision = results.box.p.mean()  # 取所有类别的平均精确率
recall = results.box.r.mean()     # 取所有类别的平均召回率
inference_time = results.speed['inference']
fps = 1000 / inference_time

# 计算参数量（YOLOv8官方标准写法）
total_params = sum(p.numel() for p in model.model.parameters()) / 1e6  # 转成M单位

print("\n" + "="*50)
print("           YOLOv8n Baseline 指标汇总")
print("="*50)
print(f"mAP@0.5        : {map50:.4f}")
print(f"mAP@0.5:0.95   : {map50_95:.4f}")
print(f"Precision      : {precision:.4f}")
print(f"Recall         : {recall:.4f}")
print(f"FPS            : {fps:.1f}")
print(f"参数量 Params  : {total_params:.2f} M")
print("="*50)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-obb summary (fused): 82 layers, 3,077,414 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 247.5±33.6 MB/s, size: 138.1 KB)
val: Scanning /kaggle/input/datasets/viviorangeani/corn-hand415/labels/val... 357 images, 63 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 357/357 716.0it/s 0.5s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/viviorangeani/corn-hand415/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 4.5it/s 5.1s
                   all        357       2242      0.491      0.508      0.377       0.19
Speed: 1.7ms preprocess, 4.2ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /kaggle/working/runs/obb/val

           YOLOv8n Baseline 指标汇总
mAP@0.5        : 0.3771
mAP@0.5:0.95   : 0.1903
Precision      : 0.4911
Recall         : 